# Lab: make a tiny task service easier to change

This notebook is a local-only practice space for Module 1. It uses dictionaries, lists, and small standard-library classes; it never contacts a network or writes a file. Run from a fresh kernel and keep your own prediction answers in the markdown prompts or in a separate note.

In [ ]:
import sys

assert sys.version_info >= (3, 10), 'Use Python 3.10 or newer.'
print(f'Python {sys.version.split()[0]} is ready; this lab uses only the standard library.')

## Objectives

By the end you will have:

- inspected a tangled in-memory task service;
- written characterization tests before changing its structure;
- mapped responsibilities and stated a pre-edit hypothesis;
- extracted a rule and injected a fake repository and fixed clock;
- checked valid, invalid, unauthorized, and storage-failure paths;
- critiqued an intentionally over-engineered AI-style design.

## A two-minute syntax warm-up

A function is a named recipe. `def` defines it and parentheses call it. A dictionary maps keys to values, a list holds ordered values, and `assert` checks an expectation. The next example is intentionally small so you can see the input-to-output flow before the service gets more interesting.

In [ ]:
def add_tax(price, rate):
    return price * (1 + rate)

total = add_tax(100, 0.10)
assert round(total, 2) == 110.0
print(total)

## Prediction 1 — function calls

Before running the next cell, predict: what will `recipe` contain, and what will `result` contain? Remember that a function name without parentheses refers to the recipe; parentheses run it.

In [ ]:
def double(number):
    return number * 2

recipe = double
result = double(4)
assert callable(recipe)
assert result == 8
print('recipe is callable:', callable(recipe), '| result:', result)

**Prediction 1 answer:** `recipe` is the function object, so `callable(recipe)` is true; `result` is `8` because `double(4)` runs the recipe. Passing a function as a value is one simple form of composition: another function can use it without knowing how it was written.

## Objects and explicit dependencies

A class describes objects. The object below stores one date and exposes a method. A method is a function attached to an object; `self` refers to the particular object. A dependency is something another piece needs. Passing a clock into a service is dependency injection: the caller chooses which clock to use.

In [ ]:
class FixedClock:
    def __init__(self, today):
        self.today = today

    def current_date(self):
        return self.today

clock = FixedClock('2026-01-15')
assert clock.current_date() == '2026-01-15'
print(clock.current_date())

## Prediction 2 — composition

Suppose a service receives `clock` as an argument. Predict the result of this call before running it: `stamp_title('Read lesson', clock)`. Why is this easier to test than calling the real calendar inside the function?

In [ ]:
def stamp_title(title, clock):
    return {'title': title, 'created_on': clock.current_date()}

stamped = stamp_title('Read lesson', clock)
assert stamped == {'title': 'Read lesson', 'created_on': '2026-01-15'}
print(stamped)

**Prediction 2 answer:** the result contains the title and the fixed date. The function does not know whether `clock` is a real clock or a fake; a test can pass a fixed object and avoid waiting for, or depending on, today's date. This is composition: the function is built from a collaborator it receives.

## 1. Inspect the tangled baseline

The baseline intentionally combines transport parsing, validation, business rules, storage, and response formatting. `TANGLED_TASKS` and `TANGLED_TODAY` are global values: the function reaches outward instead of receiving dependencies. Global state is acceptable here only as a disposable teaching example; it is exactly the coupling we will make visible and replace.

In [ ]:
TANGLED_TASKS = []
TANGLED_TODAY = '2026-01-15'

def tangled_create_task(request):
    title = request.get('title', '').strip()
    owner = request.get('owner', '').strip()
    if not title or not owner:
        return {'status': 400, 'error': 'title and owner are required'}
    if len(title) > 80:
        return {'status': 400, 'error': 'title is too long'}
    task = {
        'id': len(TANGLED_TASKS) + 1,
        'title': title,
        'owner': owner,
        'status': 'pending',
        'created_on': TANGLED_TODAY,
    }
    TANGLED_TASKS.append(task)
    return {'status': 201, 'task': task}

def tangled_complete_task(task_id, actor):
    task = next((item for item in TANGLED_TASKS if item['id'] == task_id), None)
    if task is None:
        return {'status': 404, 'error': 'task not found'}
    if task['owner'] != actor:
        return {'status': 403, 'error': 'owner required'}
    task['status'] = 'completed'
    return {'status': 200, 'task': task}

TANGLED_TASKS.clear()
baseline_created = tangled_create_task({'title': 'Read lesson', 'owner': 'ada'})
assert baseline_created['status'] == 201
assert baseline_created['task']['created_on'] == '2026-01-15'
print(baseline_created)

## 2. Characterization tests

A characterization test records what the existing code does before a structural change. These tests are not yet a design improvement; they are a safety net. We reset the synthetic global list before each small scenario so one example cannot hide state from another.

In [ ]:
def reset_tangled():
    TANGLED_TASKS.clear()

reset_tangled()
created = tangled_create_task({'title': 'Write tests', 'owner': 'ada'})
assert created == {
    'status': 201,
    'task': {
        'id': 1, 'title': 'Write tests', 'owner': 'ada',
        'status': 'pending', 'created_on': '2026-01-15'
    },
}

missing = tangled_create_task({'title': 'No owner'})
assert missing == {'status': 400, 'error': 'title and owner are required'}

unknown = tangled_complete_task(999, 'ada')
assert unknown == {'status': 404, 'error': 'task not found'}

forbidden = tangled_complete_task(1, 'grace')
assert forbidden == {'status': 403, 'error': 'owner required'}

completed = tangled_complete_task(1, 'ada')
assert completed['status'] == 200
assert completed['task']['status'] == 'completed'
print('Characterization checks passed:', 5)

## 3. Responsibility map and hypothesis

Write this map in your own words before changing the code. Then write a falsifiable hypothesis beginning with **If** and ending with an observable result. Example: “If validation becomes a plain function and storage/time are passed into the core operation, the same task responses will pass while the rule can be tested without global state.”

In [ ]:
responsibility_map = {
    'transport parsing': 'tangled_create_task reads title and owner',
    'business validation': 'tangled_create_task checks required fields and title length',
    'storage and ids': 'TANGLED_TASKS plus len(...)',
    'time': 'TANGLED_TODAY',
    'response formatting': 'tangled_create_task and tangled_complete_task choose status/error shapes',
}
hypothesis = (
    'If the rule is extracted and store/clock are explicit, '    'the public responses will stay the same and focused tests will need no globals.'
)
assert set(responsibility_map) == {
    'transport parsing', 'business validation', 'storage and ids', 'time', 'response formatting'
}
print('Map recorded.')
print(hypothesis)

## 4. Incremental extraction

First extract the rule. It returns an error message or `None`; it does not know about HTTP status codes. A repository is the component responsible for storing and retrieving records; our in-memory repository is a fake used only for this lab. Then create that repository and a clock. These are deliberately boring collaborators: boring code is easy to inspect.

In [ ]:
def validate_task(title, owner):
    title = title.strip()
    owner = owner.strip()
    if not title or not owner:
        return 'title and owner are required'
    if len(title) > 80:
        return 'title is too long'
    return None

class InMemoryTaskRepository:
    def __init__(self, fail_on_save=False):
        self.items = []
        self.fail_on_save = fail_on_save

    def next_id(self):
        return len(self.items) + 1

    def save(self, task):
        if self.fail_on_save:
            raise RuntimeError('storage unavailable')
        self.items.append(task.copy())

    def get(self, task_id):
        return next((item for item in self.items if item['id'] == task_id), None)

    def update(self, task):
        existing = self.get(task['id'])
        if existing is None:
            raise KeyError(task['id'])
        existing.update(task)

class FixedClock:
    def __init__(self, today):
        self.today = today

    def current_date(self):
        return self.today

In [ ]:
def create_task_record(title, owner, repository, clock):
    error = validate_task(title, owner)
    if error:
        return {'ok': False, 'kind': 'validation', 'error': error}
    task = {
        'id': repository.next_id(),
        'title': title.strip(),
        'owner': owner.strip(),
        'status': 'pending',
        'created_on': clock.current_date(),
    }
    try:
        repository.save(task)
    except RuntimeError as exc:
        return {'ok': False, 'kind': 'storage', 'error': str(exc)}
    return {'ok': True, 'task': task}

def complete_task_record(task_id, actor, repository):
    task = repository.get(task_id)
    if task is None:
        return {'ok': False, 'kind': 'not_found', 'error': 'task not found'}
    if task['owner'] != actor:
        return {'ok': False, 'kind': 'forbidden', 'error': 'owner required'}
    task['status'] = 'completed'
    repository.update(task)
    return {'ok': True, 'task': task}

def handle_create(request, repository, clock):
    result = create_task_record(request.get('title', ''), request.get('owner', ''), repository, clock)
    if result['ok']:
        return {'status': 201, 'task': result['task']}
    status = 503 if result['kind'] == 'storage' else 400
    return {'status': status, 'error': result['error']}

def handle_complete(task_id, actor, repository):
    result = complete_task_record(task_id, actor, repository)
    if result['ok']:
        return {'status': 200, 'task': result['task']}
    status_by_kind = {'not_found': 404, 'forbidden': 403}
    return {'status': status_by_kind[result['kind']], 'error': result['error']}

## Prediction 3 — failure boundaries

Before running the next checks, predict the status returned when a repository is configured to fail on save. Should a validation error be reported as a storage error? The core operation knows the kind of failure; the handler maps it to a transport status.

In [ ]:
repository = InMemoryTaskRepository()
clock = FixedClock('2026-01-15')
valid = handle_create({'title': 'Read lesson', 'owner': 'ada'}, repository, clock)
invalid = handle_create({'title': '', 'owner': 'ada'}, repository, clock)
unknown = handle_complete(999, 'ada', repository)
forbidden = handle_complete(1, 'grace', repository)

failing_repository = InMemoryTaskRepository(fail_on_save=True)
failure = handle_create({'title': 'Save me', 'owner': 'ada'}, failing_repository, clock)

assert valid['status'] == 201
assert invalid == {'status': 400, 'error': 'title and owner are required'}
assert unknown == {'status': 404, 'error': 'task not found'}
assert forbidden == {'status': 403, 'error': 'owner required'}
assert failure == {'status': 503, 'error': 'storage unavailable'}
print(valid)
print(invalid)
print(unknown)
print(forbidden)
print(failure)

**Prediction 3 answer:** the failing repository produces status `503`, while invalid input produces `400`. The core operation labels the failure, and the handler translates that label into an external response. Catching only the expected storage exception keeps unrelated programming errors visible instead of hiding them.

## Positive, negative, and failure-path evidence

The positive path is a valid creation and completion. Negative paths are expected rejections such as missing input, unknown task, and wrong owner. The failure path is a dependency problem. All three matter: happy-path code can look correct while error mapping or replacement boundaries are broken.

In [ ]:
completed = handle_complete(1, 'ada', repository)
assert completed['status'] == 200
assert completed['task']['status'] == 'completed'
assert repository.get(1)['status'] == 'completed'

assert validate_task('  ', 'ada') == 'title and owner are required'
assert validate_task('x' * 81, 'ada') == 'title is too long'
assert validate_task('Useful task', 'ada') is None
print('Positive, negative, and failure-path checks passed.')

## AI-generated design to critique (do not execute)

Imagine an AI tool proposed the design below for this tiny lab. It is shown only as text so it cannot change your notebook. Identify at least two unnecessary abstractions and one risk before reading the sample critique.

```python
class AbstractTaskRepositoryFactoryProvider:
    def build_repository(self, configuration):
        return RepositoryFactory(configuration).create()

class GenericServiceContainer:
    def resolve(self, name):
        ...

class TaskApplicationOrchestrator(BaseApplicationOrchestrator):
    def run(self, context):
        return self.container.resolve('task_service').execute(context)
```

### Sample critique

None of the classes has a current consumer in this notebook; the direct repository and clock arguments already provide the needed composition. The generic container hides dependencies and makes it harder to follow which objects are used. The inheritance hierarchy adds parent behavior without a demonstrated “is-a” need. The ellipsis is also not an implementation, so the proposed design cannot be tested as shown. A useful AI review records that the smallest working boundary is already enough.

## Guided TODO — attempt before opening the solution

Write a function named `task_label(task)` in your notes or an empty scratch cell. It should return `'#<id>: <title> (<status>)'`, for example `'#1: Read lesson (pending)'`. Do not change the task or add a class. Think about which responsibility this function owns and what it should *not* do. After trying, run the reference solution below.

In [ ]:
def task_label(task):
    return f"#{task['id']}: {task['title']} ({task['status']})"

label = task_label({'id': 7, 'title': 'Review diff', 'status': 'pending'})
assert label == '#7: Review diff (pending)'
print(label)

The reference solution has one narrow responsibility: formatting a task for display. It does not save, validate, or change the task. A different implementation using local variables would also be correct if it preserves that boundary.

## Independent challenge

Add a `list_for_owner(owner, repository)` function in a scratch cell. It should return new list values containing only tasks whose `owner` matches, without modifying repository state. Write one positive test, one empty-result test, and one test proving that the original task dictionaries are unchanged. Before coding, state whether this belongs in the handler, rule layer, or repository and why. You may use the same composition and fake-dependency ideas from the lesson.

## Exit questions and answers

Answer these without looking back, then compare with the answers.

1. What behavior did the characterization tests protect?
2. Why is `repository` an explicit dependency?
3. What does the `503` failure-path check prove, and what does it not prove?
4. Give one reason to leave a small duplication in place.
5. What would you record when an AI tool generates a refactor?

### Answers

1. They protected the baseline's returned statuses, error messages, task fields, and owner-completion behavior while the internal arrangement changed.
2. The core operation needs storage to choose an ID and save a record, and the caller supplies which implementation to use. Tests can therefore pass a fresh in-memory or failing repository.
3. It proves this local handler maps the synthetic repository's expected save exception to status `503`. It does not prove reliability of a real database, retries, concurrency, or network behavior.
4. Similar code may belong to different responsibilities or change for different reasons; sharing it could create a misleading abstraction and tighter coupling.
5. Record the prompt and response, inspect every changed line and import, run meaningful behavior tests, verify referenced APIs, and note suggestions you rejected or could not justify.

## Evidence handoff

Save or report: the baseline and final representative outputs, characterization and focused test results, your responsibility map and hypothesis, the positive/negative/failure checks, and your AI diff-review notes. Restart the kernel and run all cells once more. This local notebook proves a small seam and stable synthetic behavior; it does not prove production safety, database concurrency, or HTTP framework integration.